In [53]:
pip install pandas openpyxl requests statsmodels

In [54]:
import pandas as pd
import requests
import numpy as np
import gdown  # Librería mandatoria para bypass de Google Drive API
from io import BytesIO, StringIO
from statsmodels.tsa.arima.model import ARIMA

In [55]:
# =========================================================================
# 1. DESCARGA AUTOMÁTICA Y REAL DE TODAS LAS FUENTES (100% REMOTO)
# =========================================================================

# URL 1: Pasajeros Subte Mensual (DGEYC)
url_pasajeros = "https://www.estadisticaciudad.gob.ar/eyc/wp-content/uploads/2026/03/TR_FSM_AX03.xlsx"

# URL 2: Histórico SMVM Mensual (datos.gob.ar)
url_smvm = "https://infra.datos.gob.ar/catalog/sspm/dataset/57/distribution/57.1/download/indice-salario-minimo-vital-movil-valores-mensuales-pesos-corrientes-desde-1988.csv"

# URL 3: ID del archivo de tarifas completo en Google Drive y endpoint de descarga universal
id_archivo_tarifas = "1-rScuLTAgNOEFwCEetVSnShEe1DUjbwJ"
url_tarifas_drive = f"https://docs.google.com/uc?export=download&id={id_archivo_tarifas}"



print("Iniciando descarga global de datasets macroeconómicos...")

# A) Descarga de Pasajeros (.xlsx de la Ciudad)
res_pasajeros = requests.get(url_pasajeros)
if res_pasajeros.status_code == 200:
    df_pasajeros_crudo = pd.read_excel(BytesIO(res_pasajeros.content), sheet_name=0)
    print("-> Dataset de pasajeros TR_FSM_AX03 cargado con éxito.")
else:
    raise Exception("Error al descargar el archivo de pasajeros de la DGEYC.")

# B) Descarga de SMVM (.csv oficial de datos.gob.ar)
res_smvm = requests.get(url_smvm)
if res_smvm.status_code == 200:
    csv_smvm_data = StringIO(res_smvm.content.decode('utf-8'))
    df_smvm_crudo = pd.read_csv(csv_smvm_data)
    print("-> Dataset de SMVM cargado con éxito.")
else:
    raise Exception("Error al descargar el CSV del SMVM de datos.gob.ar.")

# C) Descarga REAL y DINÁMICA del CSV desde Google Drive usando Requests Universal
print("-> Conectando de forma dinámica con tu CSV de tarifas en Google Drive...")
res_tarifas = requests.get(url_tarifas_drive)
if res_tarifas.status_code == 200:
    texto_csv = res_tarifas.content.decode('utf-8')
    df_tarifas_crudo = pd.read_csv(StringIO(texto_csv))
    print("-> Dataset de tarifas desde Google Drive sincronizado dinámicamente con éxito.")
else:
    raise Exception(f"No se pudo conectar con Drive. Código de error del servidor: {res_tarifas.status_code}")

Iniciando descarga global de datasets macroeconómicos...
-> Dataset de pasajeros TR_FSM_AX03 cargado con éxito.
-> Dataset de SMVM cargado con éxito.
-> Conectando de forma dinámica con tu CSV de tarifas en Google Drive...
-> Dataset de tarifas desde Google Drive sincronizado dinámicamente con éxito.


In [56]:
# =========================================================================
# 2. PROCESAMIENTO COMPLEJO DE PASAJEROS (REPORT FORMAT)
# =========================================================================
print("\nProcesando y reconstruyendo la estructura temporal de pasajeros...")

for i in range(15):
    df_prueba = pd.read_excel(BytesIO(res_pasajeros.content), sheet_name=0, skiprows=i)
    columnas_limpias = [str(c).strip() for c in df_prueba.columns]
    if any(x in columnas_limpias for x in ['Mes', 'Meses', 'Periodo']):
        df_pasajeros_crudo = df_prueba.copy()
        break

nuevos_encabezados = ['Mes', 'Total', 'Linea A', 'Linea B', 'Linea C', 'Linea D', 'Linea E', 'Linea H', 'Premetro']
df_pasajeros_crudo.columns = nuevos_encabezados + list(df_pasajeros_crudo.columns[9:])

meses_map = {
    'Enero': 1, 'Febrero': 2, 'Marzo': 3, 'Abril': 4, 'Mayo': 5, 'Junio': 6,
    'Julio': 7, 'Agosto': 8, 'Septiembre': 9, 'Octubre': 10, 'Noviembre': 11, 'Diciembre': 12
}

anio_actual = None
fechas_reconstruidas = []
filas_a_mantener = []

for idx, row in df_pasajeros_crudo.iterrows():
    valor_mes = str(row['Mes']).strip()
    if valor_mes.isdigit() and len(valor_mes) == 4:
        anio_actual = int(valor_mes)
        continue
    if valor_mes in meses_map and anio_actual is not None:
        num_mes = meses_map[valor_mes]
        fechas_reconstruidas.append(pd.Timestamp(year=anio_actual, month=num_mes, day=1))
        filas_a_mantener.append(idx)

df_pasajeros_limpio = df_pasajeros_crudo.loc[filas_a_mantener].copy()
df_pasajeros_limpio['fecha'] = fechas_reconstruidas

for col in nuevos_encabezados[1:]:
    if df_pasajeros_limpio[col].dtype == 'object':
        df_pasajeros_limpio[col] = df_pasajeros_limpio[col].astype(str).str.replace('.', '', regex=False).str.strip()
    df_pasajeros_limpio[col] = pd.to_numeric(df_pasajeros_limpio[col], errors='coerce')

df_pasajeros_final = df_pasajeros_limpio[
    (df_pasajeros_limpio['fecha'] >= '2014-01-01') & (df_pasajeros_limpio['fecha'] <= '2019-12-31')
].copy()

df_pasajeros_final['total_subte_limpio'] = df_pasajeros_final['Total'] - df_pasajeros_final['Premetro']

# Eliminamos físicamente la columna 'Premetro' del dataframe
df_pasajeros_final.drop(columns=['Premetro'], inplace=True)


Procesando y reconstruyendo la estructura temporal de pasajeros...


In [57]:
# =========================================================================
# 3. PROCESAMIENTO Y ALINEACIÓN DE TU CSV DE TARIFAS REAL
# =========================================================================
print("Procesando y acotando el dataset de tarifas de subte...")

# Limpiamos espacios en blanco invisibles de los encabezados actuales
df_tarifas_crudo.columns = [str(c).strip() for c in df_tarifas_crudo.columns]

# Protegemos el código renombrando las columnas por su posición física en el CSV
df_tarifas_crudo.rename(columns={
    df_tarifas_crudo.columns[0]: 'año',
    df_tarifas_crudo.columns[1]: 'mes_numero',
    df_tarifas_crudo.columns[3]: 'precio'
}, inplace=True)

# UNIFICACIÓN DE COLUMNAS: Combinamos 'año' y 'mes_numero' en una sola fecha temporal
df_tarifas_crudo['fecha'] = pd.to_datetime(
    df_tarifas_crudo['año'].astype(str) + '-' + df_tarifas_crudo['mes_numero'].astype(str) + '-01'
)

# Resample para asegurar inercia temporal si faltara registrar algún mes plano
df_tarifas_completo = df_tarifas_crudo.set_index('fecha').resample('MS').ffill().reset_index()

df_tarifas_final = df_tarifas_completo[['fecha', 'precio']].copy()
df_tarifas_final.rename(columns={'precio': 'precio_pasaje'}, inplace=True)
df_tarifas_final = df_tarifas_final[
    (df_tarifas_final['fecha'] >= '2014-01-01') & (df_tarifas_final['fecha'] <= '2019-12-31')
].copy()

Procesando y acotando el dataset de tarifas de subte...


In [58]:
# =========================================================================
# 4. PROCESAMIENTO DEL SMVM Y CRUCE INTEGRAL DE VARIABLES (MERGE)
# =========================================================================
print("Alineando matriz macroeconómica unificada...")

df_smvm_crudo['fecha'] = pd.to_datetime(df_smvm_crudo['indice_tiempo'])
df_smvm = df_smvm_crudo[['fecha', 'salario_minimo_vital_movil_mensual']].copy()
df_smvm.rename(columns={'salario_minimo_vital_movil_mensual': 'SMVM'}, inplace=True)
df_smvm = df_smvm[(df_smvm['fecha'] >= '2014-01-01') & (df_smvm['fecha'] <= '2019-12-31')].copy()

# Cruces indexados estrictos
df_macro = pd.merge(df_pasajeros_final, df_tarifas_final, on='fecha', how='left')
df_macro = pd.merge(df_macro, df_smvm, on='fecha', how='left')

# Imputaciones automáticas por descalces menores de días en extremos de bases oficiales
df_macro['precio_pasaje'] = df_macro['precio_pasaje'].ffill().bfill()
df_macro['SMVM'] = df_macro['SMVM'].ffill().bfill()

# Cálculo macroeconómico de elasticidad (50 viajes mensuales vs SMVM)
df_macro['impacto_tarifa'] = (df_macro['precio_pasaje'] * 50) / df_macro['SMVM']

# Sanitización metodológica obligatoria para evitar NaNs o Infs en statsmodels
df_macro['impacto_tarifa'] = df_macro['impacto_tarifa'].replace([np.inf, -np.inf], np.nan)
df_macro['impacto_tarifa'] = df_macro['impacto_tarifa'].ffill().bfill()

lineas_subte = ['Linea A', 'Linea B', 'Linea C', 'Linea D', 'Linea E', 'Linea H']
for linea in lineas_subte:
    df_macro[linea] = df_macro[linea].ffill().bfill()

df_macro = df_macro.sort_values('fecha')
df_macro.set_index('fecha', inplace=True)
df_macro = df_macro.asfreq('MS')

print(f"-> Unificación exitosa: {len(df_macro)} meses perfectamente alineados y limpios.")

Alineando matriz macroeconómica unificada...
-> Unificación exitosa: 72 meses perfectamente alineados y limpios.


In [59]:
# =========================================================================
# 5. CONFIGURACIÓN Y ENTRENAMIENTO DEL ARIMAX POR LÍNEA DE SUBTE
# =========================================================================
X_train = df_macro[['impacto_tarifa']]
p, d, q = 1, 1, 1
modelos_por_linea = {}

print("\nIniciando estimación de coeficientes autorregresivos...")

for linea in lineas_subte:
    print(f"\n" + "="*60)
    print(f" AJUSTANDO ARIMAX REAL: {linea.upper()} (2014-2019)")
    print("="*60)

    y_train_linea = df_macro[linea]

    model = ARIMA(y_train_linea, exog=X_train, order=(p, d, q))
    results = model.fit()

    modelos_por_linea[linea] = results
    print(results.summary().tables[1])

print("\n¡Flujo completado! Todos los modelos ARIMAX mensuales reales han sido calibrados de forma uniforme.")


Iniciando estimación de coeficientes autorregresivos...

 AJUSTANDO ARIMAX REAL: LINEA A (2014-2019)
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
impacto_tarifa -2.353e+06   3.19e-10  -7.39e+15      0.000   -2.35e+06   -2.35e+06
ar.L1             -0.5158      0.795     -0.649      0.516      -2.073       1.042
ma.L1              0.4086      0.834      0.490      0.624      -1.225       2.043
sigma2          2.191e+11    1.2e-11   1.83e+22      0.000    2.19e+11    2.19e+11

 AJUSTANDO ARIMAX REAL: LINEA B (2014-2019)
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
impacto_tarifa -3.044e+06    6.4e-09  -4.76e+14      0.000   -3.04e+06   -3.04e+06
ar.L1             -0.5636      0.887     -0.635      0.525      -2.302       1.175
ma.L1              0.48

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
impacto_tarifa -1.888e+06   3.24e-10  -5.83e+15      0.000   -1.89e+06   -1.89e+06
ar.L1              0.1203      0.898      0.134      0.893      -1.639       1.880
ma.L1             -0.0105      0.879     -0.012      0.990      -1.733       1.712
sigma2          6.035e+11   8.95e-12   6.74e+22      0.000    6.03e+11    6.03e+11

 AJUSTANDO ARIMAX REAL: LINEA E (2014-2019)
                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
impacto_tarifa  -3.18e+06   4.64e-09  -6.86e+14      0.000   -3.18e+06   -3.18e+06
ar.L1              0.1672      1.024      0.163      0.870      -1.839       2.174
ma.L1             -0.0752      1.018     -0.074      0.941      -2.070       1.920
sigma2          4.351e+10   1.38e-10   3.1

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'


                     coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------
impacto_tarifa -4.118e+06   1.24e-07  -3.31e+13      0.000   -4.12e+06   -4.12e+06
ar.L1             -0.4962      3.125     -0.159      0.874      -6.622       5.629
ma.L1              0.4668      3.189      0.146      0.884      -5.783       6.717
sigma2          4.465e+10   5.39e-10   8.29e+19      0.000    4.47e+10    4.47e+10

¡Flujo completado! Todos los modelos ARIMAX mensuales reales han sido calibrados de forma uniforme.
